# ECG-Centric PCOS Screening System

Complete Google Colab notebook for validated-feature ECG-centric PCOS screening.

This notebook runs 9 experiments across ECG-only, physiology-only, and combined feature sets using Random Forest, XGBoost, and RBF-SVM with stratified 5-fold cross-validation. It exports publication-quality plots at 300 dpi, model comparison tables, SHAP explainability outputs, feature stability reports, decision curve analysis, overfitting diagnostics, final model selection, and an executive report.

**Input file expected:** `best_bayesian_150.csv`

Columns `Subject_ID` and `Diagnosis` are ignored if present. The target is `Target_Class`.


In [ ]:

# ================================================================
# ECG-Centric PCOS Screening System - Complete Executable Notebook
# ================================================================
# This cell installs/imports dependencies and configures plotting.

import importlib.util
import os
import sys
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "xgboost": "xgboost",
    "shap": "shap",
}

missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module_name) is None]
if missing:
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import json
import math
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier
import shap

RANDOM_STATE = 42
N_SPLITS = 5
INPUT_CSV = Path("best_bayesian_150.csv")
OUTPUT_DIR = Path("ecg_pcos_screening_outputs")

EXPORT_DIRS = {
    "confusion_matrices": Path("confusion_matrices"),
    "roc_curves": Path("roc_curves"),
    "pr_curves": Path("pr_curves"),
    "learning_curves": Path("learning_curves"),
    "calibration_curves": Path("calibration_curves"),
    "shap": Path("shap"),
    "decision_curve_analysis": Path("decision_curve_analysis"),
    "feature_stability": Path("feature_stability"),
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for directory in EXPORT_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.titlesize": 13,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
})

print("Environment ready.")
print(f"Outputs will be saved to: {OUTPUT_DIR.resolve()}")


In [ ]:

# ================================================================
# Load and validate dataset
# ================================================================
# In Google Colab, place best_bayesian_150.csv in /content or upload it when prompted.

if not INPUT_CSV.exists():
    try:
        from google.colab import files
        print("best_bayesian_150.csv was not found in the current Colab session.")
        print("Please upload best_bayesian_150.csv when prompted.")
        uploaded = files.upload()
        if "best_bayesian_150.csv" not in uploaded:
            first_uploaded = next(iter(uploaded.keys()))
            Path(first_uploaded).rename(INPUT_CSV)
            print(f"Renamed uploaded file {first_uploaded} -> {INPUT_CSV}")
    except Exception as exc:
        raise FileNotFoundError(
            "best_bayesian_150.csv not found. Upload it to the notebook working directory."
        ) from exc

raw_df = pd.read_csv(INPUT_CSV)
print(f"Loaded {INPUT_CSV}: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
print(raw_df.head())

# Canonical feature names requested by the project. The mapping accepts common unit/spacing variants.
def normalize_name(name):
    return "".join(ch.lower() for ch in str(name) if ch.isalnum())

CANONICAL_ALIASES = {
    "HRV_SDNN_Estimate": ["HRV_SDNN_Estimate", "HRV_SDNN_Estimate (ms)", "HRV SDNN Estimate", "HRV SDNN Estimate (ms)"],
    "P_Wave_Dispersion": ["P_Wave_Dispersion", "P_Wave_Dispersion (s)", "P Wave Dispersion", "P Wave Dispersion (s)"],
    "SDNN_MeanRR_Ratio": ["SDNN_MeanRR_Ratio", "SDNN MeanRR Ratio", "SDNN Mean RR Ratio"],
    "QTc_Bazett": ["QTc_Bazett", "QTc_Bazett (s)", "QTc Bazett", "QTc Bazett (s)"],
    "PR_Interval": ["PR_Interval", "PR_Interval (s)", "PR Interval", "PR Interval (s)"],
    "T_Wave_Duration": ["T_Wave_Duration", "T_Wave_Duration (s)", "T Wave Duration", "T Wave Duration (s)"],
    "Cycle_Length": ["Cycle_Length", "Cycle Length", "Cycle_Length (days)", "Cycle Length (days)"],
    "Cycle_Regularity": ["Cycle_Regularity", "Cycle Regularity"],
    "BMI": ["BMI", "Body Mass Index"],
    "Family_History": ["Family_History", "Family History", "Family History (PCOS/Diabetes)", "Family_History (PCOS/Diabetes)"],
    "Target_Class": ["Target_Class", "Target Class"],
    "Subject_ID": ["Subject_ID", "Subject ID"],
    "Diagnosis": ["Diagnosis"],
}

normalized_to_column = {normalize_name(col): col for col in raw_df.columns}
rename_map = {}
missing_alias_groups = []
for canonical, aliases in CANONICAL_ALIASES.items():
    matched = None
    for alias in aliases:
        key = normalize_name(alias)
        if key in normalized_to_column:
            matched = normalized_to_column[key]
            break
    if matched is not None:
        rename_map[matched] = canonical
    elif canonical in ["Target_Class"]:
        missing_alias_groups.append(canonical)

df = raw_df.rename(columns=rename_map).copy()

# Explicitly ignore columns requested by the project.
columns_to_ignore = [col for col in ["Subject_ID", "Diagnosis"] if col in df.columns]
if columns_to_ignore:
    df = df.drop(columns=columns_to_ignore)
    print(f"Ignored columns: {columns_to_ignore}")

if "Target_Class" not in df.columns:
    raise ValueError("Target_Class is required but was not found after canonical column mapping.")

ECG_FEATURES = [
    "HRV_SDNN_Estimate",
    "P_Wave_Dispersion",
    "SDNN_MeanRR_Ratio",
    "QTc_Bazett",
    "PR_Interval",
    "T_Wave_Duration",
]
PHYSIOLOGY_FEATURES = [
    "Cycle_Length",
    "Cycle_Regularity",
    "BMI",
    "Family_History",
]
COMBINED_FEATURES = ECG_FEATURES + PHYSIOLOGY_FEATURES
FEATURE_SETS = {
    "ECG Only": ECG_FEATURES,
    "Physiology Only": PHYSIOLOGY_FEATURES,
    "Combined": COMBINED_FEATURES,
}

missing_features = [feature for feature in COMBINED_FEATURES if feature not in df.columns]
if missing_features:
    raise ValueError(f"Missing required validated features: {missing_features}")

# Keep only final validated feature set plus target.
df = df[COMBINED_FEATURES + ["Target_Class"]].copy()

# Drop rows with missing target and encode target with PCOS as the positive class when present.
df = df.dropna(subset=["Target_Class"]).reset_index(drop=True)
target_text = df["Target_Class"].astype(str)
if target_text.str.lower().str.contains("pcos").any():
    y = target_text.str.lower().str.contains("pcos").astype(int).to_numpy()
    target_mapping = {"negative_or_normal": 0, "pcos": 1}
else:
    encoder = LabelEncoder()
    y = encoder.fit_transform(target_text)
    if len(np.unique(y)) != 2:
        raise ValueError("Target_Class must be binary for ROC-AUC, PR-AUC, and decision curve analysis.")
    target_mapping = {str(cls): int(idx) for idx, cls in enumerate(encoder.classes_)}

X_all = df[COMBINED_FEATURES].copy()
print("Target mapping:", target_mapping)
print("Class counts:")
print(pd.Series(y, name="Target").value_counts().sort_index())
print("Validated features used:", COMBINED_FEATURES)


In [ ]:

# ================================================================
# Modeling, evaluation, plotting, explainability, and utility helpers
# ================================================================

def safe_slug(text):
    return str(text).replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "").replace("-", "_")


def make_preprocessor(X):
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
    categorical_cols = [col for col in X.columns if col not in numeric_cols]
    try:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", SimpleImputer(strategy="median"), numeric_cols),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", cat_encoder),
            ]), categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor, numeric_cols, categorical_cols


def get_processed_feature_names(preprocessor, numeric_cols, categorical_cols):
    names = list(numeric_cols)
    if categorical_cols:
        onehot = preprocessor.named_transformers_["categorical"].named_steps["onehot"]
        names.extend(onehot.get_feature_names_out(categorical_cols).tolist())
    return names


def make_models(scale_for_svm=False):
    return {
        "Random Forest": RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=350,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "SVM (RBF Kernel)": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    }


def build_pipeline(model_name, estimator, X):
    preprocessor, _, _ = make_preprocessor(X)
    if model_name == "SVM (RBF Kernel)":
        return Pipeline([
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", estimator),
        ])
    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator),
    ])


def predict_probability(estimator, X):
    if hasattr(estimator, "predict_proba"):
        return estimator.predict_proba(X)[:, 1]
    scores = estimator.decision_function(X)
    return 1.0 / (1.0 + np.exp(-scores))


def compute_metric_dict(y_true, y_pred, y_prob):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
    }


def plot_confusion_matrix(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5.5, 4.8), constrained_layout=True)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Non-PCOS", "PCOS"], yticklabels=["Non-PCOS", "PCOS"], ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_roc_curve(y_true, y_prob, title, path):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_value = roc_auc_score(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    ax.plot(fpr, tpr, color="#1f77b4", lw=2, label=f"AUC = {auc_value:.3f}")
    ax.plot([0, 1], [0, 1], color="#6c757d", lw=1.5, linestyle="--", label="Chance")
    ax.set_title(title)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.legend(loc="lower right", frameon=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_pr_curve(y_true, y_prob, title, path):
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    prevalence = np.mean(y_true)
    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    ax.plot(recall, precision, color="#2a9d8f", lw=2, label=f"PR-AUC = {ap:.3f}")
    ax.axhline(prevalence, color="#6c757d", lw=1.5, linestyle="--", label=f"Prevalence = {prevalence:.3f}")
    ax.set_title(title)
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_ylim(0, 1.05)
    ax.legend(loc="lower left", frameon=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_calibration_curve(y_true, y_prob, title, path):
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=min(10, max(3, len(y_true)//15)), strategy="quantile")
    fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
    ax.plot(mean_pred, frac_pos, marker="o", lw=2, color="#8f5f2f", label="Model")
    ax.plot([0, 1], [0, 1], linestyle="--", color="#6c757d", label="Perfect calibration")
    ax.set_title(title)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed PCOS fraction")
    ax.legend(loc="upper left", frameon=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_learning_curve(estimator, X, y, title, path):
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    train_sizes = np.linspace(0.25, 1.0, 5)
    sizes, train_scores, val_scores = learning_curve(
        estimator, X, y,
        cv=cv,
        train_sizes=train_sizes,
        scoring="roc_auc",
        n_jobs=-1,
        shuffle=True,
        random_state=RANDOM_STATE,
    )
    train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
    val_mean, val_std = val_scores.mean(axis=1), val_scores.std(axis=1)
    fig, ax = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
    ax.plot(sizes, train_mean, marker="o", color="#264653", label="Training ROC-AUC")
    ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, color="#264653", alpha=0.18)
    ax.plot(sizes, val_mean, marker="o", color="#e76f51", label="Validation ROC-AUC")
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, color="#e76f51", alpha=0.18)
    ax.set_title(title)
    ax.set_xlabel("Training samples")
    ax.set_ylabel("ROC-AUC")
    ax.set_ylim(0.0, 1.05)
    ax.legend(loc="lower right", frameon=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return {
        "train_final_mean": float(train_mean[-1]),
        "train_final_std": float(train_std[-1]),
        "validation_final_mean": float(val_mean[-1]),
        "validation_final_std": float(val_std[-1]),
        "generalization_gap": float(train_mean[-1] - val_mean[-1]),
    }


def interpret_overfitting(lc):
    train_auc = lc["train_final_mean"]
    val_auc = lc["validation_final_mean"]
    gap = lc["generalization_gap"]
    bias = "Low" if val_auc >= 0.85 else ("Moderate" if val_auc >= 0.70 else "High")
    variance = "Low" if gap <= 0.05 else ("Moderate" if gap <= 0.12 else "High")
    if gap > 0.12 and train_auc >= 0.85:
        risk = "High"
    elif gap > 0.05:
        risk = "Moderate"
    else:
        risk = "Low"
    return bias, variance, risk


def decision_curve_values(y_true, y_prob, thresholds):
    n = len(y_true)
    prevalence = np.mean(y_true)
    rows = []
    for threshold in thresholds:
        pred = (y_prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        net_benefit = (tp / n) - (fp / n) * (threshold / (1 - threshold))
        treat_all = prevalence - (1 - prevalence) * (threshold / (1 - threshold))
        rows.append({
            "threshold": threshold,
            "model_net_benefit": net_benefit,
            "treat_all_net_benefit": treat_all,
            "treat_none_net_benefit": 0.0,
        })
    return pd.DataFrame(rows)


def plot_decision_curve(dca_df, title, path):
    fig, ax = plt.subplots(figsize=(6.5, 5), constrained_layout=True)
    ax.plot(dca_df["threshold"], dca_df["model_net_benefit"], color="#1f77b4", lw=2, label="Model")
    ax.plot(dca_df["threshold"], dca_df["treat_all_net_benefit"], color="#9b3d3d", lw=1.7, linestyle="--", label="Treat all")
    ax.plot(dca_df["threshold"], dca_df["treat_none_net_benefit"], color="#6c757d", lw=1.7, linestyle=":", label="Treat none")
    ax.set_title(title)
    ax.set_xlabel("Threshold probability")
    ax.set_ylabel("Net benefit")
    ax.legend(loc="best", frameon=True)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def extract_fold_importance(fitted_pipeline, model_name, X_valid, y_valid, feature_names):
    model = fitted_pipeline.named_steps["model"]
    if model_name in ["Random Forest", "XGBoost"] and hasattr(model, "feature_importances_"):
        values = np.asarray(model.feature_importances_, dtype=float)
        return pd.Series(values, index=feature_names)

    # For SVM, use permutation importance on the full pipeline for original features.
    perm = permutation_importance(
        fitted_pipeline, X_valid, y_valid,
        scoring="roc_auc",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    return pd.Series(perm.importances_mean, index=X_valid.columns)


def aggregate_original_feature_importance(importance_series, original_features):
    out = pd.Series(0.0, index=original_features)
    for name, value in importance_series.items():
        matched = False
        for feature in original_features:
            if name == feature or name.startswith(feature + "_"):
                out[feature] += float(value)
                matched = True
                break
        if not matched and name in out.index:
            out[name] += float(value)
    total = out.abs().sum()
    if total > 0:
        out = out / total
    return out


In [ ]:

# ================================================================
# Run 9 experiments with stratified 5-fold cross-validation
# ================================================================

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
models = make_models()

summary_rows = []
fold_metric_rows = []
oof_predictions = {}
learning_curve_rows = []
stability_records = []

for feature_set_name, feature_list in FEATURE_SETS.items():
    X = X_all[feature_list].copy()
    for model_name, estimator in models.items():
        experiment_id = f"{safe_slug(model_name)}__{safe_slug(feature_set_name)}"
        print(f"Running: {model_name} | {feature_set_name}")

        y_oof_prob = np.zeros(len(y), dtype=float)
        y_oof_pred = np.zeros(len(y), dtype=int)
        fold_metrics = []

        for fold_idx, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y[train_idx], y[valid_idx]

            pipeline = build_pipeline(model_name, clone(estimator), X_train)
            pipeline.fit(X_train, y_train)
            y_prob = predict_probability(pipeline, X_valid)
            y_pred = (y_prob >= 0.5).astype(int)

            y_oof_prob[valid_idx] = y_prob
            y_oof_pred[valid_idx] = y_pred

            metrics = compute_metric_dict(y_valid, y_pred, y_prob)
            metrics.update({"Model": model_name, "Feature Set": feature_set_name, "Fold": fold_idx})
            fold_metrics.append(metrics)
            fold_metric_rows.append(metrics)

            fitted_preprocessor = pipeline.named_steps["preprocessor"]
            numeric_cols = [col for col in X_train.columns if pd.api.types.is_numeric_dtype(X_train[col])]
            categorical_cols = [col for col in X_train.columns if col not in numeric_cols]
            processed_feature_names = get_processed_feature_names(fitted_preprocessor, numeric_cols, categorical_cols)
            raw_importance = extract_fold_importance(pipeline, model_name, X_valid, y_valid, processed_feature_names)
            original_importance = aggregate_original_feature_importance(raw_importance, feature_list)
            for feature, importance in original_importance.items():
                stability_records.append({
                    "Model": model_name,
                    "Feature Set": feature_set_name,
                    "Fold": fold_idx,
                    "Feature": feature,
                    "Importance": float(importance),
                })

        fold_df = pd.DataFrame(fold_metrics)
        row = {"Model": model_name, "Feature Set": feature_set_name, "Experiment ID": experiment_id}
        for metric in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]:
            row[metric] = fold_df[metric].mean()
            row[f"{metric} SD"] = fold_df[metric].std(ddof=1)
            row[f"{metric} Mean ± SD"] = f"{row[metric]:.3f} ± {row[f'{metric} SD']:.3f}"
        summary_rows.append(row)
        oof_predictions[experiment_id] = {
            "Model": model_name,
            "Feature Set": feature_set_name,
            "features": feature_list,
            "y_true": y.copy(),
            "y_prob": y_oof_prob.copy(),
            "y_pred": y_oof_pred.copy(),
        }

        plot_confusion_matrix(
            y, y_oof_pred,
            f"Confusion Matrix - {model_name} ({feature_set_name})",
            EXPORT_DIRS["confusion_matrices"] / f"{experiment_id}.png",
        )
        plot_roc_curve(
            y, y_oof_prob,
            f"ROC Curve - {model_name} ({feature_set_name})",
            EXPORT_DIRS["roc_curves"] / f"{experiment_id}.png",
        )
        plot_pr_curve(
            y, y_oof_prob,
            f"Precision-Recall Curve - {model_name} ({feature_set_name})",
            EXPORT_DIRS["pr_curves"] / f"{experiment_id}.png",
        )
        plot_calibration_curve(
            y, y_oof_prob,
            f"Calibration Curve - {model_name} ({feature_set_name})",
            EXPORT_DIRS["calibration_curves"] / f"{experiment_id}.png",
        )

        full_pipeline = build_pipeline(model_name, clone(estimator), X)
        lc = plot_learning_curve(
            full_pipeline, X, y,
            f"Learning Curve - {model_name} ({feature_set_name})",
            EXPORT_DIRS["learning_curves"] / f"{experiment_id}.png",
        )
        bias, variance, risk = interpret_overfitting(lc)
        learning_curve_rows.append({
            "Model": model_name,
            "Feature Set": feature_set_name,
            **lc,
            "Bias": bias,
            "Variance": variance,
            "Overfitting Risk": risk,
        })

        dca_df = decision_curve_values(y, y_oof_prob, thresholds=np.linspace(0.01, 0.99, 99))
        dca_df.insert(0, "Model", model_name)
        dca_df.insert(1, "Feature Set", feature_set_name)
        dca_df.to_csv(EXPORT_DIRS["decision_curve_analysis"] / f"{experiment_id}.csv", index=False)
        plot_decision_curve(
            dca_df,
            f"Decision Curve Analysis - {model_name} ({feature_set_name})",
            EXPORT_DIRS["decision_curve_analysis"] / f"{experiment_id}.png",
        )

fold_metrics_df = pd.DataFrame(fold_metric_rows)
model_comparison = pd.DataFrame(summary_rows)
model_comparison = model_comparison.sort_values(
    ["ROC-AUC", "PR-AUC", "F1", "Accuracy"], ascending=False
).reset_index(drop=True)
model_comparison.insert(0, "Rank", np.arange(1, len(model_comparison) + 1))

ranking_columns = ["Rank", "Model", "Feature Set", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
model_comparison[ranking_columns].to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
fold_metrics_df.to_csv(OUTPUT_DIR / "cross_validation_fold_metrics.csv", index=False)

learning_curve_report = pd.DataFrame(learning_curve_rows)
learning_curve_report.to_csv(OUTPUT_DIR / "overfitting_analysis.csv", index=False)

print("\nModel comparison ranking:")
display(model_comparison[ranking_columns])
print("\nOverfitting analysis:")
display(learning_curve_report)


In [ ]:

# ================================================================
# SHAP explainability and feature stability reports
# ================================================================

def get_shap_matrix(shap_values):
    if isinstance(shap_values, list):
        return shap_values[1] if len(shap_values) > 1 else shap_values[0]
    arr = np.asarray(shap_values)
    if arr.ndim == 3:
        return arr[:, :, 1] if arr.shape[2] > 1 else arr[:, :, 0]
    return arr


def create_shap_outputs(model_name, feature_set_name, estimator, feature_list):
    X = X_all[feature_list].copy()
    experiment_id = f"{safe_slug(model_name)}__{safe_slug(feature_set_name)}"
    pipeline = build_pipeline(model_name, clone(estimator), X)
    pipeline.fit(X, y)

    preprocessor = pipeline.named_steps["preprocessor"]
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col])]
    categorical_cols = [col for col in X.columns if col not in numeric_cols]
    feature_names = get_processed_feature_names(preprocessor, numeric_cols, categorical_cols)
    X_processed = preprocessor.transform(X)
    X_processed_df = pd.DataFrame(X_processed, columns=feature_names)
    model = pipeline.named_steps["model"]

    explainer = shap.TreeExplainer(model)
    shap_values = get_shap_matrix(explainer.shap_values(X_processed_df))
    shap_values = np.asarray(shap_values)

    # SHAP summary plot.
    plt.figure(figsize=(8, max(4.8, 0.35 * len(feature_names))))
    shap.summary_plot(shap_values, X_processed_df, show=False, plot_size=None)
    plt.title(f"SHAP Summary - {model_name} ({feature_set_name})")
    plt.tight_layout()
    plt.savefig(EXPORT_DIRS["shap"] / f"{experiment_id}_summary.png", dpi=300, bbox_inches="tight")
    plt.close()

    mean_abs = np.abs(shap_values).mean(axis=0)
    shap_importance = pd.DataFrame({"Feature": feature_names, "Mean |SHAP|": mean_abs})
    shap_importance = shap_importance.sort_values("Mean |SHAP|", ascending=False)
    shap_importance.to_csv(EXPORT_DIRS["shap"] / f"{experiment_id}_feature_importance.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, max(4.5, 0.35 * len(shap_importance))), constrained_layout=True)
    sns.barplot(data=shap_importance.head(20).sort_values("Mean |SHAP|"),
                x="Mean |SHAP|", y="Feature", color="#2f6f8f", ax=ax)
    ax.set_title(f"SHAP Feature Importance - {model_name} ({feature_set_name})")
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_ylabel("")
    fig.savefig(EXPORT_DIRS["shap"] / f"{experiment_id}_feature_importance.png", bbox_inches="tight")
    plt.close(fig)

    top_feature = shap_importance.iloc[0]["Feature"]
    plt.figure(figsize=(6.5, 5))
    shap.dependence_plot(top_feature, shap_values, X_processed_df, show=False)
    plt.title(f"SHAP Dependence - {top_feature}\n{model_name} ({feature_set_name})")
    plt.tight_layout()
    plt.savefig(EXPORT_DIRS["shap"] / f"{experiment_id}_dependence_{safe_slug(top_feature)}.png", dpi=300, bbox_inches="tight")
    plt.close()

    return shap_importance.assign(Model=model_name, **{"Feature Set": feature_set_name})

shap_importance_frames = []
for feature_set_name, feature_list in FEATURE_SETS.items():
    for model_name in ["Random Forest", "XGBoost"]:
        print(f"Generating SHAP outputs: {model_name} | {feature_set_name}")
        try:
            shap_importance_frames.append(
                create_shap_outputs(model_name, feature_set_name, models[model_name], feature_list)
            )
        except Exception as exc:
            print(f"SHAP generation failed for {model_name} | {feature_set_name}: {exc}")

if shap_importance_frames:
    all_shap_importance = pd.concat(shap_importance_frames, ignore_index=True)
    all_shap_importance.to_csv(EXPORT_DIRS["shap"] / "all_shap_feature_importance.csv", index=False)

stability_df = pd.DataFrame(stability_records)
stability_summary = (
    stability_df.groupby(["Model", "Feature Set", "Feature"], as_index=False)
    .agg(
        Mean_Importance=("Importance", "mean"),
        SD_Importance=("Importance", "std"),
        CV_Importance=("Importance", lambda x: float(np.std(x, ddof=1) / np.mean(x)) if np.mean(x) != 0 else np.nan),
    )
)
stability_summary["Stability"] = np.where(
    stability_summary["CV_Importance"].fillna(np.inf) <= 0.5, "High",
    np.where(stability_summary["CV_Importance"].fillna(np.inf) <= 1.0, "Moderate", "Low")
)
stability_summary = stability_summary.sort_values(
    ["Model", "Feature Set", "Mean_Importance"], ascending=[True, True, False]
)
stability_df.to_csv(EXPORT_DIRS["feature_stability"] / "feature_importance_by_fold.csv", index=False)
stability_summary.to_csv(EXPORT_DIRS["feature_stability"] / "feature_stability_report.csv", index=False)

for (model_name, feature_set_name), group in stability_summary.groupby(["Model", "Feature Set"]):
    experiment_id = f"{safe_slug(model_name)}__{safe_slug(feature_set_name)}"
    fig, ax = plt.subplots(figsize=(8, max(4.5, 0.35 * len(group))), constrained_layout=True)
    plot_group = group.sort_values("Mean_Importance", ascending=True)
    ax.barh(plot_group["Feature"], plot_group["Mean_Importance"],
            xerr=plot_group["SD_Importance"].fillna(0), color="#3f7f6f", alpha=0.9)
    ax.set_title(f"Feature Stability - {model_name} ({feature_set_name})")
    ax.set_xlabel("Mean normalized importance across CV folds")
    ax.set_ylabel("")
    fig.savefig(EXPORT_DIRS["feature_stability"] / f"{experiment_id}_feature_stability.png", bbox_inches="tight")
    plt.close(fig)

print("Feature stability report:")
display(stability_summary)


In [ ]:

# ================================================================
# Final model selection, clinical interpretation, and exports
# ================================================================

selection_basis = model_comparison.copy()

def best_for_feature_set(name):
    return selection_basis[selection_basis["Feature Set"] == name].sort_values(
        ["ROC-AUC", "PR-AUC", "F1", "Accuracy"], ascending=False
    ).iloc[0]

best_ecg = best_for_feature_set("ECG Only")
best_phys = best_for_feature_set("Physiology Only")
best_combined = best_for_feature_set("Combined")
overall_winner = selection_basis.iloc[0]

final_model_report = pd.DataFrame([
    {"Selection": "Best ECG-Only Model", **best_ecg.to_dict()},
    {"Selection": "Best Physiology-Only Model", **best_phys.to_dict()},
    {"Selection": "Best Combined Model", **best_combined.to_dict()},
    {"Selection": "Overall Winner", **overall_winner.to_dict()},
])
final_model_report.to_csv(OUTPUT_DIR / "final_model_report.csv", index=False)

winner_model = overall_winner["Model"]
winner_feature_set = overall_winner["Feature Set"]
winner_features = FEATURE_SETS[winner_feature_set]

winner_stability = stability_summary[
    (stability_summary["Model"] == winner_model) &
    (stability_summary["Feature Set"] == winner_feature_set)
].sort_values("Mean_Importance", ascending=False)
top_biomarkers = winner_stability["Feature"].head(5).tolist()

clinical_notes = []
if any(f in top_biomarkers for f in ECG_FEATURES):
    clinical_notes.append(
        "ECG-derived autonomic and repolarization markers contribute materially to PCOS discrimination."
    )
if any(f in top_biomarkers for f in PHYSIOLOGY_FEATURES):
    clinical_notes.append(
        "Physiological and reproductive-history variables add clinically interpretable context."
    )
if winner_feature_set == "Combined":
    clinical_notes.append(
        "The combined model supports a multimodal screening strategy rather than relying on ECG or physiology alone."
    )
else:
    clinical_notes.append(
        f"The best-performing screening signal came from the {winner_feature_set} feature set."
    )
clinical_interpretation = " ".join(clinical_notes)

deployment_recommendation = (
    "Use as a research-grade screening aid only after external validation, calibration assessment, "
    "and prospective evaluation against site-specific prevalence and workflow requirements."
)

ready = (
    overall_winner["ROC-AUC"] >= 0.80 and
    overall_winner["PR-AUC"] >= 0.80 and
    learning_curve_report[
        (learning_curve_report["Model"] == winner_model) &
        (learning_curve_report["Feature Set"] == winner_feature_set)
    ].iloc[0]["Overfitting Risk"] != "High"
)
final_statement = "READY FOR EXTERNAL VALIDATION" if ready else "FURTHER OPTIMIZATION REQUIRED"

executive_report = f"""
ECG-CENTRIC PCOS SCREENING SYSTEM - EXECUTIVE MODEL REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

1. Best Model
{winner_model}

2. Best Feature Set
{winner_feature_set}

3. ROC-AUC
{overall_winner['ROC-AUC']:.3f} ± {overall_winner['ROC-AUC SD']:.3f}

4. PR-AUC
{overall_winner['PR-AUC']:.3f} ± {overall_winner['PR-AUC SD']:.3f}

5. Top Biomarkers
{', '.join(top_biomarkers) if top_biomarkers else 'Not available'}

6. Clinical Interpretation
{clinical_interpretation}

7. Deployment Recommendation
{deployment_recommendation}

Final statement:
{final_statement}
""".strip()

(OUTPUT_DIR / "executive_model_report.txt").write_text(executive_report)

# Also copy key files to the notebook root with the exact requested names.
shutil.copy2(OUTPUT_DIR / "model_comparison.csv", Path("model_comparison.csv"))
shutil.copy2(OUTPUT_DIR / "final_model_report.csv", Path("final_model_report.csv"))
shutil.copy2(OUTPUT_DIR / "executive_model_report.txt", Path("executive_model_report.txt"))

print(executive_report)
print("\nSaved exports:")
print(f"- {OUTPUT_DIR / 'model_comparison.csv'}")
print(f"- {OUTPUT_DIR / 'final_model_report.csv'}")
print(f"- {OUTPUT_DIR / 'executive_model_report.txt'}")
for name, directory in EXPORT_DIRS.items():
    print(f"- {directory}/")
